# Step 3: Time-aware Disease Labeling + Comorbidity Extraction (CAD vs Non-CAD + CVD vs Non-CVD)

**Input:** `master-demographic-ppg-data-no-icd.csv` + `diagnosis-events-for-time-aware-labeling.csv` (from Step 2)

This notebook:
1. Builds two disease classification tasks (CAD vs Non-CAD, CVD vs Non-CVD) with temporal safeguards to reduce label leakage
2. Extracts comorbidity flags (diabetes, high cholesterol, obesity) from diagnosis events

## Label Definitions

### CAD vs Non-CAD (Strict)
1. **CAD (label=1):** ICD-9 CAD code evidence using MIMIC-III prefix 4140 present and valid by time rule.
2. **Non-CAD (label=0):** No cardiovascular disease evidence by strict control rule.
3. **Unresolved:** Rows that cannot be confidently assigned (kept for review).

### CVD vs Non-CVD (Binary)
1. **CVD (label=1):** Any implemented cardiovascular disease evidence using ICD-9 prefixes 410-414, 420-429, and 440-448 by time rule.
2. **Non-CVD (label=0):** No heart-related disease evidence.

## Key Principles
1. Use diagnosis events at admission level (HADM_ID) and admission time.
2. Prefer event-time-aware labels when a PPG timestamp is available.
3. If no PPG timestamp exists, use fallback encounter-level strict labeling.
4. Keep strict controls for Non-CAD: exclude any cardiovascular disease.
5. CVD is an exhaustive binary classification (every row is either CVD or Non-CVD).

**Features Extracted:**
- **Disease labels:** CAD classification (`label`), CVD classification (`CVD_LABEL`)
- **Comorbidity flags:** `is_diabetic` (ICD-250*), `has_high_cholesterol` (ICD-272*), `is_obese` (ICD-278*)

**Outputs:** 
- PPG data with demographics + disease labels + comorbidity flags → feeds to Step 4

In [ ]:
import os
from pathlib import Path
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
BASE_DIR = PROJECT_ROOT / "data" / "intermediate"

PPG_FILE = BASE_DIR / "master-demographic-ppg-data-no-icd.csv"
DIAG_EVENTS_FILE = BASE_DIR / "diagnosis-events-for-time-aware-labeling.csv"
ADMISSIONS_FILE = PROJECT_ROOT / "data" / "mimic_clinical" / "ADMISSIONS.csv"

OUT_FULL = BASE_DIR / "PPG_cad_flagged_timeaware_strict.csv"
OUT_CAD = BASE_DIR / "CAD-patients-data-timeaware-strict.csv"
OUT_NONCAD = BASE_DIR / "Non-CAD-patients-data-timeaware-strict.csv"
OUT_REVIEW = BASE_DIR / "PPG_unresolved-labels-timeaware-strict.csv"

OUT_CVD_FULL = BASE_DIR / "PPG_cvd_flagged_timeaware.csv"
OUT_CVD = BASE_DIR / "CVD-patients-data-timeaware.csv"
OUT_NONCVD = BASE_DIR / "Non-CVD-patients-data-timeaware.csv"


# Strict CAD vs Non-CAD policy
CAD_PREFIXES = ("4140",)
HEART_PREFIXES = (
    "410", "411", "412", "413", "414",
    "420", "421", "422", "423", "424", "425", "426", "427", "428", "429",
    "440", "441", "442", "443", "444", "445", "446", "447", "448"
)


## **Import Libraries and Configure File Paths**

In [ ]:
# ----------------------------
# Load and validate inputs
# ----------------------------
ppg_df = pd.read_csv(PPG_FILE)
diag_df = pd.read_csv(DIAG_EVENTS_FILE)
adm_df = pd.read_csv(ADMISSIONS_FILE)

print(f"PPG rows: {len(ppg_df)}")
print(f"Diagnosis event rows: {len(diag_df)}")
print(f"Admissions rows: {len(adm_df)}")

required_ppg = {"SUBJECT_ID", "patient_id", "record_id"}
required_diag = {"SUBJECT_ID", "HADM_ID", "ICD9_CODE"}
required_adm = {"SUBJECT_ID", "HADM_ID", "ADMITTIME"}

miss_ppg = sorted(required_ppg - set(ppg_df.columns))
miss_diag = sorted(required_diag - set(diag_df.columns))
miss_adm = sorted(required_adm - set(adm_df.columns))

if miss_ppg or miss_diag or miss_adm:
    raise ValueError(
        f"Missing columns -> PPG: {miss_ppg}, DIAG: {miss_diag}, ADMISSIONS: {miss_adm}"
    )

# Optional PPG time columns (for strict time-aware mode)
ppg_time_candidates = ["PPG_TIME", "ppg_time", "record_time", "CHARTTIME", "timestamp"]
ppg_time_col = next((c for c in ppg_time_candidates if c in ppg_df.columns), None)

if ppg_time_col is None:
    print("No PPG timestamp column found. Notebook will use fallback encounter-level labeling.")
else:
    print(f"PPG timestamp column detected: {ppg_time_col}")

## **Load and Validate Input Datasets**

Verify all required columns are present in PPG data, diagnosis events, and admissions tables.

In [ ]:
# ----------------------------
# Helper functions
# ----------------------------
def normalize_icd9(code):
    if pd.isna(code):
        return ""
    s = str(code).strip().upper()
    return s.replace(".", "")

def is_cad_icd9(code):
    c = normalize_icd9(code)
    return any(c.startswith(prefix) for prefix in CAD_PREFIXES)

def is_heart_icd9(code):
    c = normalize_icd9(code)
    return any(c.startswith(prefix) for prefix in HEART_PREFIXES)

## **Define ICD-9 Classification Functions**

Helper functions to normalize ICD-9 codes and identify CAD vs heart disease categories.

In [ ]:
# ----------------------------
# Build diagnosis events with event time
# ----------------------------
adm_small = adm_df[["SUBJECT_ID", "HADM_ID", "ADMITTIME"]].copy()
adm_small["ADMITTIME"] = pd.to_datetime(adm_small["ADMITTIME"], errors="coerce")

events = diag_df[["SUBJECT_ID", "HADM_ID", "SEQ_NUM", "ICD9_CODE"]].copy()
events = events.merge(adm_small, on=["SUBJECT_ID", "HADM_ID"], how="left")
events = events.dropna(subset=["SUBJECT_ID", "ICD9_CODE", "ADMITTIME"])

events["ICD9_NORM"] = events["ICD9_CODE"].apply(normalize_icd9)
events["is_cad"] = events["ICD9_CODE"].apply(is_cad_icd9)
events["is_heart"] = events["ICD9_CODE"].apply(is_heart_icd9)

print(f"Usable diagnosis events: {len(events)}")
print(f"CAD events: {events['is_cad'].sum()}")
print(f"Heart-related events: {events['is_heart'].sum()}")

## **Build Diagnosis Event Timeline**

Merge admission times with diagnosis events to create time-ordered event history for each patient.

In [ ]:
# ----------------------------
# Derive patient-level event summaries and labels
# ----------------------------
cad_first = (
    events[events["is_cad"]]
    .groupby("SUBJECT_ID", as_index=False)["ADMITTIME"]
    .min()
    .rename(columns={"ADMITTIME": "first_cad_time"})
)

heart_first = (
    events[events["is_heart"]]
    .groupby("SUBJECT_ID", as_index=False)["ADMITTIME"]
    .min()
    .rename(columns={"ADMITTIME": "first_heart_time"})
)

ppg_labeled = ppg_df.copy()
ppg_labeled = ppg_labeled.merge(cad_first, on="SUBJECT_ID", how="left")
ppg_labeled = ppg_labeled.merge(heart_first, on="SUBJECT_ID", how="left")

if ppg_time_col is not None:
    ppg_labeled["ppg_time"] = pd.to_datetime(ppg_labeled[ppg_time_col], errors="coerce")
    labeling_mode = "time-aware strict (using PPG timestamp)"

    # CAD only if CAD evidence exists before or at PPG time.
    ppg_labeled["is_cad"] = (
        ppg_labeled["first_cad_time"].notna() &
        ppg_labeled["ppg_time"].notna() &
        (ppg_labeled["first_cad_time"] <= ppg_labeled["ppg_time"])
)

    # Strict Non-CAD: no heart-disease evidence at or before PPG time.
    ppg_labeled["is_non_cad"] = (
        ~ppg_labeled["is_cad"] &
        (ppg_labeled["first_heart_time"].isna() | (ppg_labeled["first_heart_time"] > ppg_labeled["ppg_time"]))
)

    # CVD: any heart-related disease at or before PPG time.
    ppg_labeled["is_cvd"] = (
        ppg_labeled["first_heart_time"].notna() &
        ppg_labeled["ppg_time"].notna() &
        (ppg_labeled["first_heart_time"] <= ppg_labeled["ppg_time"])
)
else:
    # Fallback strict mode when PPG timestamp is unavailable.
    labeling_mode = "fallback encounter-level strict (no PPG timestamp found)"

    # CAD if any CAD evidence exists in available encounters.
    ppg_labeled["is_cad"] = ppg_labeled["first_cad_time"].notna()

    # Strict Non-CAD: no heart-disease evidence in available encounters.
    ppg_labeled["is_non_cad"] = ppg_labeled["first_heart_time"].isna()

    # CVD: any heart-related disease in available encounters.
    ppg_labeled["is_cvd"] = ppg_labeled["first_heart_time"].notna()

# Unresolved rows are those that satisfy neither strict CAD condition.
ppg_labeled["label"] = np.where(
    ppg_labeled["is_cad"],
    1,
    np.where(ppg_labeled["is_non_cad"], 0, np.nan)
)

# CVD label is exhaustive binary: CVD=1, Non-CVD=0.
ppg_labeled["CVD_LABEL"] = ppg_labeled["is_cvd"].astype(int)

print("Labeling mode:", labeling_mode)
print("CAD rows:", int((ppg_labeled["label"] == 1).sum()))
print("Non-CAD rows:", int((ppg_labeled["label"] == 0).sum()))
print("Unresolved rows:", int(ppg_labeled["label"].isna().sum()))
print("CVD rows:", int((ppg_labeled["CVD_LABEL"] == 1).sum()))
print("Non-CVD rows:", int((ppg_labeled["CVD_LABEL"] == 0).sum()))

## **Derive Patient-Level Labels (CAD/Non-CAD and CVD/Non-CVD)**

Apply time-aware or fallback labeling rules based on diagnosis event timing relative to PPG recording time.

## **Extract Comorbidity Flags from Diagnosis Events**

Use the same diagnosis events to extract comorbidity indicators (diabetes, high cholesterol, obesity).


In [ ]:
# ----------------------------
# Extract comorbidity flags from diagnosis events
# ----------------------------

# Group diagnosis events by SUBJECT_ID and aggregate normalized ICD codes
comorbidity_data = {}

for subject_id in events["SUBJECT_ID"].unique():
    patient_events = events[events["SUBJECT_ID"] == subject_id]
    normalized_codes = patient_events["ICD9_NORM"].tolist()
    
    # Check for comorbidity indicators
    is_diabetic = int(any(code.startswith("250") for code in normalized_codes if pd.notna(code)))
    has_high_cholesterol = int(any(code.startswith("272") for code in normalized_codes if pd.notna(code)))
    is_obese = int(any(code.startswith("278") for code in normalized_codes if pd.notna(code)))
    
    comorbidity_data[subject_id] = {
        "is_diabetic": is_diabetic,
        "has_high_cholesterol": has_high_cholesterol,
        "is_obese": is_obese
    }

# Create comorbidity dataframe and merge with labeled data
comorbidity_df = pd.DataFrame(comorbidity_data).T.reset_index()
comorbidity_df.columns = ["SUBJECT_ID", "is_diabetic", "has_high_cholesterol", "is_obese"]
comorbidity_df["SUBJECT_ID"] = comorbidity_df["SUBJECT_ID"].astype(int)

# Merge into ppg_labeled
ppg_labeled = ppg_labeled.merge(
    comorbidity_df[["SUBJECT_ID", "is_diabetic", "has_high_cholesterol", "is_obese"]],
    on="SUBJECT_ID",
    how="left"
)

# Fill NaN with 0 (if any subject had no diagnosis records)
for col in ["is_diabetic", "has_high_cholesterol", "is_obese"]:
    ppg_labeled[col] = ppg_labeled[col].fillna(0).astype(int)

print(f"Comorbidity flags extracted and merged:")
print(ppg_labeled[["SUBJECT_ID", "is_diabetic", "has_high_cholesterol", "is_obese"]].head())
print(f"Comorbidity summary:")
print(f"  Diabetic: {ppg_labeled['is_diabetic'].sum()} / {len(ppg_labeled)}")
print(f"  High cholesterol: {ppg_labeled['has_high_cholesterol'].sum()} / {len(ppg_labeled)}")
print(f"  Obese: {ppg_labeled['is_obese'].sum()} / {len(ppg_labeled)}")

In [ ]:
# ----------------------------
# Basic quality checks
# ----------------------------
# Overlap check: no row should be both CAD and Non-CAD
overlap = (ppg_labeled["is_cad"] & ppg_labeled["is_non_cad"]).sum()
print(f"Overlap rows (must be 0): {int(overlap)}")

# CVD completeness check: every row should be either CVD or Non-CVD
cvd_completeness = ((ppg_labeled["CVD_LABEL"] == 0) | (ppg_labeled["CVD_LABEL"] == 1)).all()
print(f"CVD label is complete binary (must be True): {cvd_completeness}")

# Subject-level sanity for CAD strict labels
subject_summary = ppg_labeled.groupby("SUBJECT_ID")["label"].agg(["count", "nunique"]).reset_index()
mixed_subjects = (subject_summary["nunique"] > 1).sum()
print(f"Subjects with mixed labels across rows: {int(mixed_subjects)}")

ppg_labeled[["patient_id", "SUBJECT_ID", "record_id", "label", "CVD_LABEL", "is_cad", "is_non_cad", "is_cvd"]].head()

## **Quality Assurance: Validate Label Consistency**

Check for label overlaps, completeness, and subject-level consistency.

In [ ]:
# ----------------------------
# Save outputs
# ----------------------------
cad_df = ppg_labeled[ppg_labeled["label"] == 1].copy()
noncad_df = ppg_labeled[ppg_labeled["label"] == 0].copy()
review_df = ppg_labeled[ppg_labeled["label"].isna()].copy()
cvd_df = ppg_labeled[ppg_labeled["CVD_LABEL"] == 1].copy()
noncvd_df = ppg_labeled[ppg_labeled["CVD_LABEL"] == 0].copy()

ppg_labeled.to_csv(OUT_FULL, index=False)
cad_df.to_csv(OUT_CAD, index=False)
noncad_df.to_csv(OUT_NONCAD, index=False)
review_df.to_csv(OUT_REVIEW, index=False)
ppg_labeled.to_csv(OUT_CVD_FULL, index=False)
cvd_df.to_csv(OUT_CVD, index=False)
noncvd_df.to_csv(OUT_NONCVD, index=False)

print("Saved files:")
print(f"  Full labeled (CAD strict): {OUT_FULL} ({len(ppg_labeled)} rows)")
print(f"  CAD only: {OUT_CAD} ({len(cad_df)} rows)")
print(f"  Non-CAD only: {OUT_NONCAD} ({len(noncad_df)} rows)")
print(f"  Unresolved for review: {OUT_REVIEW} ({len(review_df)} rows)")
print(f"  Full labeled (CVD binary): {OUT_CVD_FULL} ({len(ppg_labeled)} rows)")
print(f"  CVD only: {OUT_CVD} ({len(cvd_df)} rows)")
print(f"  Non-CVD only: {OUT_NONCVD} ({len(noncvd_df)} rows)")

## **Export Labeled Datasets**

Save separate files for CAD, Non-CAD, CVD, and Non-CVD datasets, plus unresolved rows for review.

## Notes
- This notebook is configured for strict CAD vs Non-CAD only.
- Non-CAD controls exclude any cardiovascular disease evidence.
- Preferred mode is true time-aware labeling using a real PPG timestamp column.
- If no PPG timestamp exists, fallback encounter-level strict labeling is used and must be reported as a limitation.
- Unresolved rows are preserved for audit instead of force-labeling.